In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
#load data and clean
#TODO: Find ways to clean the data with missing values
data = pd.read_csv('ds_challenge_v2_1_data.csv')
data['driver_signup'] = data['first_completed_date'].notnull().astype(int)
X = data.drop(['first_completed_date', 'driver_signup'], axis=1)
y = data['driver_signup']

categorical_features = ['city_name', 'signup_os', 'signup_channel', 'vehicle_make', 'vehicle_model']

# Convert all date columns to numerical features
# calculate days since these dates occured
date_cols = ['signup_date', 'bgc_date', 'vehicle_added_date']
for col in date_cols:
    X[col] = pd.to_datetime(X[col], format='%m/%d/%y', errors='coerce')
    X[col] = X[col].fillna(pd.Timestamp.min)
    X[col + '_year'] = X[col].dt.year
    X[col + '_month'] = X[col].dt.month
    X[col + '_day'] = X[col].dt.day
    X = X.drop(col, axis=1)

# Impute missing values using SimpleImputer before OneHotEncoding
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')  # Use most frequent for categorical features
X[categorical_features] = imputer.fit_transform(X[categorical_features])

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_features = encoder.fit_transform(X[categorical_features])

encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_features))

X = X.drop(categorical_features, axis=1)
X = pd.concat([X, encoded_df], axis=1)

# Impute missing numerical values after OneHotEncoding
for col in [col for col in X.columns if col.endswith(('_year', '_month', '_day'))]:
    X[col] = X[col].fillna(0)  # Fill NaNs with 0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

#temporary, not sure if this accuracy would be very helpful.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8793087684008412


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## **What to do with model?**

We want to input certain values depending of the feature, and output how likely someone is to start driving given certain values. For example, determine how likely one is to start driving for Uber if they signed up in Berton compared to Starak.

## Tim Rework Below:

In [ ]:
df = pd.read_csv('ds_challenge_v2_1_data.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (54681, 11)


/usr/local/lib/python3.11/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")
/usr/local/lib/python3.11/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")
/usr/local/lib/python3.11/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,id,city_name,signup_os,signup_channel,signup_date,bgc_date,vehicle_added_date,vehicle_make,vehicle_model,vehicle_year,first_completed_date
0,1,Strark,ios web,Paid,1/2/16,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Strark,windows,Paid,1/21/16,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Wrouver,windows,Organic,1/11/16,1/11/16,NaN,NaN,NaN,NaN,NaN
3,4,Berton,android web,Referral,1/29/16,2/3/16,2/3/16,Toyota,Corolla,2016.0,2/3/16
4,5,Strark,android web,Referral,1/10/16,1/25/16,1/26/16,Hyundai,Sonata,2016.0,NaN


In [ ]:
df['vehicle_year'] = df['vehicle_year'].astype('float64')
date_cols = ['signup_date','first_completed_date','bgc_date', 'vehicle_added_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format = '%m/%d/%y')

df['is_driver'] = ~ pd.isna(df['first_completed_date'])

In [ ]:
# time between metrics
df['signup_to_bgc'] = (df['bgc_date'] - df['signup_date']) / datetime.timedelta(days = 1)
df['signup_to_vehicle_added'] = (df['vehicle_added_date'] - df['signup_date'])/datetime.timedelta(days = 1)
#df['signup_to_first_completed'] = (df['first_completed_date'] - df['signup_date'])/datetime.timedelta(days = 1)
df['bgc_to_vehicle_added'] = (df['vehicle_added_date'] - df['bgc_date'])/datetime.timedelta(days = 1)
df['bgc_to_first_completed'] = (df['first_completed_date'] - df['bgc_date'])/datetime.timedelta(days = 1)
#df['vehicle_added_to_first_completed'] = (df['first_completed_date'] - df['vehicle_added_date'])/datetime.timedelta(days = 1)


# snapshot date
snap_date = pd.to_datetime('3/1/16',format = '%m/%d/%y')
# days since difference metrics
df['days_since_signup'] = (snap_date - df['signup_date'])/datetime.timedelta(days = 1)
df['days_since_bgc'] = (snap_date - df['bgc_date'])/datetime.timedelta(days = 1)
df['days_since_vehicle_added'] = (snap_date - df['vehicle_added_date'])/datetime.timedelta(days = 1)
df['days_since_last_activity'] = df[['days_since_signup', 'days_since_bgc', 'days_since_vehicle_added']].min(axis = 1)

In [ ]:
X = df[['city_name', 'signup_os', 'signup_channel', 'vehicle_year', 'vehicle_make', 'vehicle_model', 'signup_to_bgc', 'signup_to_vehicle_added', 'bgc_to_vehicle_added',
        'days_since_signup', 'days_since_bgc', 'days_since_vehicle_added', 'days_since_last_activity']]
y = df['is_driver'] # label



In [ ]:
#cleaning X
categorical_features = ['city_name', 'signup_os', 'signup_channel', 'vehicle_make', 'vehicle_model']

# Impute missing values using SimpleImputer before OneHotEncoding
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')
X[categorical_features] = imputer.fit_transform(X[categorical_features])

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_features = encoder.fit_transform(X[categorical_features])

encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_features))

X = X.drop(categorical_features, axis=1)
X = pd.concat([X, encoded_df], axis=1)

num_features = X.select_dtypes(include=np.number).columns
imputer_num = SimpleImputer(strategy='median')
X[num_features] = imputer_num.fit_transform(X[num_features])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

<ipython-input-11-7dbf51459030>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[categorical_features] = imputer.fit_transform(X[categorical_features])


In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)

#temporary, not sure if this accuracy would be very helpful.

#TODO: confusion matrix, precision recall, ROC-AUC,
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9352656121422693


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
#Other metrics
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_curve, roc_auc_score, auc



# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of positive class

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", conf_matrix)

# Precision-Recall Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)


# Precision, Recall, and F1-score
from sklearn.metrics import classification_report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)
print("\nROC-AUC:", roc_auc)



Accuracy: 0.9352656121422693

Confusion Matrix:
 [[9388  316]
 [ 392  841]]

Classification Report:
               precision    recall  f1-score   support

       False       0.96      0.97      0.96      9704
        True       0.73      0.68      0.70      1233

    accuracy                           0.94     10937
   macro avg       0.84      0.82      0.83     10937
weighted avg       0.93      0.94      0.93     10937


ROC-AUC: 0.9459319874781781


## What to do with Bryan's data:

We can use Bryan's cleaned car data to isolate cars by valid vehicles(meaning car makes with 4+ seats). In addition we can use his data to see what model of car would most likely be used.